In [ ]:


from pathlib import Path
import pandas as pd
import numpy as np


PATH_GIT = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'Census'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'

# SharePoint OneDrive paths
PATH_ORIG = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
PATH_WEIGHTS = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data' / 'Reference' / 'Weights'
PATH_SERVER = Path(r'\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data')
PATH_DATA = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data' / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Cost' / 'Cost_5 Home Ownership'

PATH_OUT = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Products' / 'Small Data Requests' / '2025' / 'Board Meetings'



In [ ]:


## Income_1 ---

df_inc_acs1 = pd.read_excel(PATH_SERVER / 'Income_1 MPO ACS1.xlsx', sheet_name='MPO')
df_inc_acs1 = df_inc_acs1[df_inc_acs1['Race_Ethnicity'] == 'All'].reset_index(drop=True)
df_inc_acs1['source'] = 'American Community Survey 1 Year Estimate'
df_inc_acs1 = df_inc_acs1[['MPO', 'Year', 'Median Household Income', 'Margin of Error', 'Margin of Error Ratio', 'Use for Reporting', 'source']]
# display(df_inc_acs1)

df_inc_dec = pd.read_csv(PATH_ORIG / 'Income_1_Counties_DEC_NoME_raw.csv')
# display(df_inc_dec)

df_pop = pd.read_excel(PATH_WEIGHTS / 'Total_Population Counties DEC.xlsx')
df_pop = df_pop[df_pop['Year'] == 2000]
# display(df_pop)


df_inc_dec = df_inc_dec[['COUNTYNAME', 'Year', 'HCT012001']].rename(columns={'COUNTYNAME':'County Name'})
df_pop = df_pop[['County Name', 'Population']]
df_inc_dec = df_inc_dec.merge(df_pop, on='County Name')

wm = lambda x: np.average(x, weights = df_inc_dec.loc[x.index, 'Population'])
df_inc_dec = df_inc_dec.groupby(['Year'], as_index=False, sort=False).agg(HCT012001=('HCT012001', wm))
# display(df_inc_dec)


df_IAF = pd.read_excel(PATH_CONFIG0 / 'CPI_IAF.xlsx', sheet_name='BLS_West')
df_IAF = df_IAF[df_IAF['Year'] == 2000]
scalar_iaf = df_IAF['IAF_2023'].values[0]

df_inc_dec['HCT012001'] = df_inc_dec['HCT012001']*scalar_iaf
df_inc_dec = df_inc_dec.rename(columns={'HCT012001':'Median Household Income'})

df_inc_dec['MPO'] = 'SACOG'
df_inc_dec['source'] = 'Decennial Census'
# display(df_inc_dec)


df_inc = pd.concat([df_inc_acs1, df_inc_dec])
df_inc = df_inc.reset_index(drop=True)

df_inc


In [ ]:


## Cost_5 ---

df_cost_acs1 = pd.read_excel(PATH_SERVER / 'Cost_5 MPO ACS1.xlsx', sheet_name='MPO')
df_cost_acs1['source'] = 'American Community Survey 1 Year Estimate'
df_cost_acs1['Percentage'] = df_cost_acs1['Percentage']/100
df_cost_acs1['Margin of Error Ratio'] = df_cost_acs1['Margin of Error Ratio']/100
df_cost_acs1 = df_cost_acs1[['MPO', 'Year', 'Race_Ethnicity', 'Variable', 'Households', 'Percentage', 'Margin of Error', 'Margin of Error Ratio', 'Use for Reporting', 'source']]
# display(df_cost_acs1)


df_cost_dec = pd.read_excel(PATH_DATA / 'Cost_5 MPO DEC.xlsx')
df_cost_dec = df_cost_dec[df_cost_dec['Year'] == 2000]
df_cost_dec['source'] = 'Decennial Census'
df_cost_dec = df_cost_dec[['MPO', 'Year', 'Race_Ethnicity', 'Variable', 'Households', 'Percentage', 'source']]
# display(df_cost_dec)


df_cost = pd.concat([df_cost_acs1, df_cost_dec])
df_cost = df_cost.reset_index(drop=True)

df_cost




In [ ]:


with pd.ExcelWriter(PATH_OUT / '2025_11.xlsx', engine='xlsxwriter') as writer:
    df_inc .to_excel(writer, index=False, sheet_name='Income_1')
    df_cost.to_excel(writer, index=False, sheet_name='Cost_5'  )


